# OMNI Vocal Pipeline - maevn1
Full song → Lead vocal → RVC Clone → Prompt AI Mastering

In [ ]:
# @title 1. Setup Environment
import os
from google.colab import drive, files
import IPython.display as ipd
import glob

print("🚀 Setting up OMNI-vocal-pipeline...")
drive.mount('/content/drive')

!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git /content/RVC
%cd /content/RVC

!pip install -q -r requirements.txt faiss-cpu
!pip install -q audio-separator torch torchaudio pydub matchering soundfile --extra-index-url https://download.pytorch.org/whl/cu121

!mkdir -p assets/hubert assets/rmvpe
!wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt -O assets/hubert/hubert_base.pt
!wget -q https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt -O assets/rmvpe/rmvpe.pt

print("✅ Setup complete!")

In [ ]:
# @title 2. Upload maevn1 Model
print("📤 Upload your maevn1.pth")
uploaded_pth = files.upload()

print("📤 Upload .index file (recommended)")
uploaded_index = files.upload()

!mkdir -p weights logs/maevn1
!mv *.pth weights/ 2>/dev/null || echo "Model moved"
!mv *.index logs/maevn1/ 2>/dev/null || echo "Index placed"

print("✅ maevn1 model loaded!")

In [ ]:
# @title 3. Process All Files in new_render_slot (Generic + Lead/Backup Split)
import os
from audio_separator.separator import Separator
from pydub import AudioSegment

render_slot = "/content/new_render_slot"
os.makedirs(render_slot, exist_ok=True)
os.makedirs("/content/isolated", exist_ok=True)

print(f"📂 Scanning {render_slot} for audio files...")
audio_files = glob.glob(f"{render_slot}/*.[wW][aA][vV]") + glob.glob(f"{render_slot}/*.[mM][pP]3")

if not audio_files:
    print("No files found. Upload songs to new_render_slot using left sidebar.")
    uploaded = files.upload()
    for fname in uploaded:
        !mv "{fname}" "{render_slot}/"
    audio_files = glob.glob(f"{render_slot}/*.[wW][aA][vV]") + glob.glob(f"{render_slot}/*.[mM][pP]3")

for input_path in audio_files:
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    print(f"\n🎵 Processing: {base_name}")
    
    # Stage 1: Vocals isolation
    separator = Separator(output_dir="/content/isolated", model_name="htdemucs_ft", device="cuda", segment_size=256)
    separator.separate(input_path)
    
    vocal_files = [f for f in os.listdir("/content/isolated") if "vocal" in f.lower()]
    if not vocal_files: continue
    
    # Stage 2: Lead vs Backup
    separator = Separator(output_dir="/content/isolated", model_name="UVR_MDXNET_KARA_2", device="cuda")
    separator.separate(f"/content/isolated/{vocal_files[0]}")
    
    lead_files = [f for f in os.listdir("/content/isolated") if "karaoke" in f.lower() or "lead" in f.lower()]
    if lead_files:
        guide_path = f"{render_slot}/guide_{base_name}.wav"
        !cp "/content/isolated/{lead_files[0]}" "{guide_path}"
        
        audio = AudioSegment.from_wav(guide_path)
        normalized = audio.normalize(headroom=3.0)
        normalized.export(guide_path, format="wav")
        
        # RVC Inference
        output_path = f"{render_slot}/maevn1_cloned_{base_name}.wav"
        !python tools/cmd/infer_cli.py --model_name "maevn1" --input_path "{guide_path}" --opt_path "{output_path}" --f0_method "rmvpe" --f0up_key 0 --index_rate 0.75 --rms_mix_rate 0.7 --protect 0.33 --device "cuda:0" --is_half True
        
        print(f"✅ Cloned: maevn1_cloned_{base_name}.wav")

print("\n✅ All files processed!")

In [ ]:
# @title 4. AI Mastering (Prompt-based)
!pip install -q matchering

import matchering as mg
mastered_slot = "/content/new_render_slot/mastered"
os.makedirs(mastered_slot, exist_ok=True)

mastering_prompt = "loud streaming master" #@param ["loud streaming master", "warm vinyl vibe", "punchy club", "clean bright pop", "balanced"]

cloned_files = glob.glob(f"{render_slot}/maevn1_cloned_*.wav")
for f in cloned_files:
    base = os.path.basename(f).replace("maevn1_cloned_", "").replace(".wav", "")
    out_path = f"{mastered_slot}/maevn1_mastered_{base}.wav"
    print(f"Mastering {base} with prompt: {mastering_prompt}")
    mg.process(target=f, reference=None, results=[mg.Result(out_path)], loudness=-14)

print("✅ Mastering complete!")

In [ ]:
# @title 5. Download All Mastered Files
mastered_files = glob.glob("/content/new_render_slot/mastered/*.wav")
for f in mastered_files:
    files.download(f)
print("✅ All mastered files downloaded!")